# OraShift — test-set predictions

Generates PostgreSQL translations for all four test splits, from **two** models:

1. **Base** — `Qwen2.5-Coder-3B-Instruct`, untouched.
2. **Fine-tuned** — the same base plus the QLoRA adapter from the training run.

Nothing is scored here. This notebook only produces predictions; `orashift eval`
scores them locally by **executing** each one against PostgreSQL and comparing with
Oracle. Keeping generation and scoring apart matters, because scoring needs both
databases and generation needs a GPU, and nothing has both.

**Fairness is the whole point of this notebook.** Both models get byte-identical
prompts, identical greedy decoding, identical output cleaning, and identical token
budgets. The only difference is whether the adapter is attached.

| Setting | Value |
| --- | --- |
| Accelerator | GPU T4 x2 |
| Internet | On |
| Data | `orashift-qlora-adapter` dataset is attached automatically |

Expected runtime: 15–30 minutes for 660 generations.

> **Run on a Kaggle T4**; the record is `results/predict_run.json`. Every stage
> asserts its assumptions and prints what it found, so a problem surfaces early.

## 1. Setup

In [ ]:
import time as _t

_start = _t.time()
!pip install -q unsloth
print(f"\ninstall took {(_t.time() - _start) / 60:.1f} minutes")

In [ ]:
import json, re, subprocess, time
from pathlib import Path

import torch

SEED = 20260101
torch.manual_seed(SEED)

assert torch.cuda.is_available(), "No GPU. Set Accelerator to 'GPU T4 x2'."
gpu = torch.cuda.get_device_properties(0)
print(f"gpu: {gpu.name}  {gpu.total_memory / 1024**3:.1f} GB  sm_{gpu.major}{gpu.minor}")

BASE_MODEL = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 448  # longest reference answer is ~378 tokens; this leaves room
BATCH_SIZE = 8


def find_adapter():
    # Discover the adapter rather than assuming a mount path. Kaggle does not
    # always mount a dataset at /kaggle/input/<slug>, and a hardcoded path
    # failed two runs even though the dataset was correctly attached.
    root = Path("/kaggle/input")
    mounted = sorted(p.name for p in root.glob("*")) if root.exists() else []
    print("mounted inputs:", mounted)
    for candidate in sorted(root.rglob("adapter_model.safetensors")):
        return candidate.parent
    raise AssertionError(
        f"no adapter_model.safetensors under {root}. Mounted: {mounted}\n"
        "Upload it with: python scripts/kaggle_run.py upload-adapter"
    )


ADAPTER_DIR = find_adapter()
print("adapter dir:", ADAPTER_DIR)
print("adapter files:", sorted(f.name for f in ADAPTER_DIR.iterdir())[:8])

## 2. The test splits

In [ ]:
REPO_DIR = Path("/kaggle/working/orashift")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/devansh173/orashift", str(REPO_DIR)],
        check=True,
    )

DATA_DIR = REPO_DIR / "data" / "dataset"
TEST_SPLITS = ["test_in_schema", "test_unseen_schema", "test_unseen_template", "test_unseen_both"]


def load_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


SPLITS = {name: load_jsonl(DATA_DIR / f"{name}.jsonl") for name in TEST_SPLITS}
EXAMPLES = [(name, ex) for name in TEST_SPLITS for ex in SPLITS[name]]

for name in TEST_SPLITS:
    print(f"{name:22} {len(SPLITS[name]):4d}")
print(f"\n{len(EXAMPLES)} examples x 2 models = {len(EXAMPLES) * 2} generations")

## 3. Prompting and output cleaning

Both are shared by the two models, deliberately, so neither can be advantaged by a
difference in how it was asked or how its answer was read.

**Cleaning is needed and is declared.** The base model has never been shown the
output format, so it tends to wrap SQL in markdown fences and add commentary. A
fenced block will not execute, and failing it for that would measure formatting
rather than translation. So the same cleaner runs over both models' output: strip
fences, drop prose lines, take the first statement, remove a trailing semicolon.

The fine-tuned model gains nothing from this — it already emits bare SQL — but
applying it to only one side would be indefensible.

In [ ]:
FENCE = re.compile(r"^\s*```(?:sql|postgresql)?\s*|\s*```\s*$", re.IGNORECASE | re.MULTILINE)
SQL_START = (
    "SELECT",
    "WITH",
    "CREATE",
    "INSERT",
    "UPDATE",
    "DELETE",
    "MERGE",
    "ALTER",
    "DROP",
    "(",
)


def build_prompt(example):
    """System + user turns only. The answer is never shown to the model."""
    return tokenizer.apply_chat_template(
        example["messages"][:2], tokenize=False, add_generation_prompt=True
    )


def clean_sql(text):
    """Pull a single executable statement out of a model response."""
    text = FENCE.sub("", text).strip()
    lines, started = [], False
    for line in text.splitlines():
        stripped = line.strip()
        if not started:
            if stripped.upper().startswith(SQL_START):
                started = True
            else:
                continue  # prose before the SQL
        if started:
            if stripped.startswith(("--", "#")) or stripped.lower().startswith(
                ("note:", "explanation:", "this ", "here ")
            ):
                break  # prose after the SQL
            lines.append(line)
    statement = "\n".join(lines).strip() if lines else text
    statement = statement.split(";")[0].strip() if ";" in statement else statement
    return " ".join(statement.split())


# A quick check that cleaning is a no-op on well-formed output.
assert clean_sql("SELECT 1") == "SELECT 1"
assert clean_sql("```sql\nSELECT 1;\n```") == "SELECT 1"
assert clean_sql("Here is the translation:\n```sql\nSELECT 1\n```\nThis works.") == "SELECT 1"
print("clean_sql checks passed")

## 4. Batched greedy generation

In [ ]:
def generate_all(model, tokenizer, examples, label):
    """Greedy, batched, left-padded. Deterministic and identical for both models."""
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    predictions, started = [], time.time()
    for offset in range(0, len(examples), BATCH_SIZE):
        batch = examples[offset : offset + BATCH_SIZE]
        prompts = [build_prompt(ex) for _split, ex in batch]
        encoded = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH
        ).to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,  # greedy: reproducible
                pad_token_id=tokenizer.pad_token_id,
            )
        for row, (split, example) in enumerate(batch):
            new_tokens = outputs[row][encoded["input_ids"].shape[1] :]
            raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
            predictions.append(
                {
                    "unit_key": example["meta"]["unit_key"],
                    "split": split,
                    "category": example["meta"]["category"],
                    "unit_type": example["meta"]["unit_type"],
                    "schema": example["meta"]["schema"],
                    "template_id": example["meta"]["template_id"],
                    "model": label,
                    "oracle_sql": example["messages"][1]["content"],
                    "reference_sql": example["messages"][2]["content"],
                    "prediction": clean_sql(raw),
                    "raw_output": raw[:2000],
                }
            )
        done = offset + len(batch)
        if done % 40 == 0 or done == len(examples):
            rate = done / (time.time() - started)
            remaining = (len(examples) - done) / rate / 60
            print(f"  {label}: {done}/{len(examples)}  ({rate:.1f}/s, ~{remaining:.1f} min left)")
    return predictions

## 5. The base model

No adapter. This is the number the fine-tune has to beat, and it is generated
through exactly the same code path.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True
)
FastLanguageModel.for_inference(model)

base_predictions = generate_all(model, tokenizer, EXAMPLES, "base")
print(f"\n{len(base_predictions)} base predictions")

print("\n--- two samples ---")
for p in base_predictions[:2]:
    print(f"\noracle    : {p['oracle_sql'][:150]}")
    print(f"reference : {p['reference_sql'][:150]}")
    print(f"prediction: {p['prediction'][:150]}")
    print(f"raw       : {p['raw_output'][:180]!r}")

In [ ]:
# Free the base model before loading the fine-tuned one; two 3B models will not
# both fit alongside their activations.
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print(f"gpu memory allocated after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 6. The fine-tuned model

Same base weights, with the adapter attached. Unsloth loads the adapter directory
directly, which also restores the tokenizer saved alongside it.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR), max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True
)
FastLanguageModel.for_inference(model)

# Confirm the adapter is actually attached. Without this, a silent fallback to the
# base weights would produce a "fine-tuned" column identical to the base one.
peft_modules = [n for n, _ in model.named_modules() if "lora" in n.lower()]
print(f"LoRA modules present: {len(peft_modules)}")
assert peft_modules, "no LoRA layers found: the adapter did not load"

finetuned_predictions = generate_all(model, tokenizer, EXAMPLES, "finetuned")
print(f"\n{len(finetuned_predictions)} fine-tuned predictions")

## 7. Save

In [ ]:
OUT = Path("/kaggle/working")


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"{path.name}: {len(rows)} rows, {path.stat().st_size / 1024:.0f} KB")


write_jsonl(base_predictions, OUT / "predictions_base.jsonl")
write_jsonl(finetuned_predictions, OUT / "predictions_finetuned.jsonl")

# A quick, text-only sense check. It is NOT the metric: scoring happens locally by
# executing each statement. Exact string match understates correctness badly,
# because many different statements return identical results.
for label, rows in [("base", base_predictions), ("finetuned", finetuned_predictions)]:
    for split in TEST_SPLITS:
        subset = [r for r in rows if r["split"] == split]
        exact = sum(
            1
            for r in subset
            if " ".join(r["prediction"].split()).lower()
            == " ".join(r["reference_sql"].split()).lower()
        )
        empty = sum(1 for r in subset if not r["prediction"].strip())
        print(f"{label:10} {split:22} exact-match {exact:3d}/{len(subset):3d}  empty {empty}")

In [ ]:
summary = {
    "base_model": BASE_MODEL,
    "adapter": str(ADAPTER_DIR),
    "decoding": "greedy",
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": BATCH_SIZE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "seed": SEED,
    "counts": {split: len(SPLITS[split]) for split in TEST_SPLITS},
    "generated": {"base": len(base_predictions), "finetuned": len(finetuned_predictions)},
}
with open(OUT / "predict_run.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)
print(json.dumps(summary, indent=2))

## What to download

- `predictions_base.jsonl`
- `predictions_finetuned.jsonl`
- `predict_run.json`

All three go into `results/`. Then `orashift eval` scores them by execution, which is
the only measurement this project treats as real.